# ACB - database to HuWise

## Ana Carolina Brandão, Junho 2026

Este script serve para preparar automaticamente para a HuWise os dados existentes numa base de dados Aiven.

O objetivo é permitir validar e organizar os dados antes de uma futura publicação na plataforma HuWise. Para isso, o script estabelece ligação à base de dados, lê as tabelas selecionadas, obtém a estrutura das respetivas colunas e gera ficheiros locais de exportação, incluindo ficheiros CSV com os dados e ficheiros JSON com os metadados e packages de preparação.

As credenciais são carregadas através de secrets do Colab, usando dinamicamente as iniciais do nome do notebook para identificar o utilizador.

Nesta fase, o script funciona apenas em modo de preparação local, não publicando nem alterando dados na HuWise.

In [1]:
## imports
##

import os
import socket
import requests
import re
import json
import shutil
import decimal
from pathlib import Path
from datetime import datetime, date

current_env = os.environ.get('CONDA_DEFAULT_ENV')
print("current_env:", current_env)

if current_env is None:
    !pip install pymysql --quiet
    !pip install clts_pcp --quiet
    !pip install pandas --quiet
    !pip install pytz --quiet

    from google.colab import userdata
    from google.colab import files

import pymysql
import pandas as pd
import clts_pcp as clts
import pytz

print("... done.")

current_env: None
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.5 MB/s eta 0:00:00
... done.


In [2]:
## Context gathering
##

tstart = clts.getts()

DEFAULT_PARAMS = {
    "verbose": True,
    "timeout": 20,
    "export_folder": "huwise_export",
    "max_rows_per_table": None
}

verbose = DEFAULT_PARAMS["verbose"]
timeout = DEFAULT_PARAMS["timeout"]
export_folder = DEFAULT_PARAMS["export_folder"]
max_rows_per_table = DEFAULT_PARAMS["max_rows_per_table"]

# Escolher aqui as tabelas para a HuWise
TABELAS_ESCOLHIDAS = [
    "catalog",
    "catalog_v2",
    "catalog_distribution",
    "catalog_internal",
    "catalog_internal_v2",
    "catalog_keyword"
]

EXPORT_DIR = Path(export_folder)
EXPORT_DIR.mkdir(exist_ok=True)

hostname = socket.gethostname()

try:
    ip = requests.get("https://api.ipify.org", timeout=5).text
except:
    ip = "unknown"

if "__file__" in globals():
    enviro = "airflow/linux"
    script = os.path.basename(__file__)
    parts = __file__.replace("\\", "/").split("/")
    channel = parts[-2] if len(parts) >= 2 else "unknown"
else:
    enviro = "jupyter"
    channel = "colab"
    script = requests.get("http://172.28.0.12:9000/api/sessions").json()[0]["name"]

# Recupera as três primeiras letras do nome do ficheiro
# Exemplo: ACB_database_to_HuWise.ipynb -> acb
match = re.match(r"([A-Za-z]{3})", script)

if not match:
    raise ValueError(f"Não foi possível identificar o utilizador a partir do nome do ficheiro: {script}")

user = match.group(1).lower()

context = f"{hostname} ({ip}) | {user} | {channel} | {script}"
clts.setcontext(context)

if verbose:
    print("context:", context)
    print("export_folder:", EXPORT_DIR.resolve())
    print("tabelas escolhidas:", TABELAS_ESCOLHIDAS)

context: 522790e7d309 (136.107.38.238) | acb | colab | ACB-database_to_HuWise.ipynb
export_folder: /content/huwise_export
tabelas escolhidas: ['catalog', 'catalog_v2', 'catalog_distribution', 'catalog_internal', 'catalog_internal_v2', 'catalog_keyword']


In [3]:
## Ler secrets da base de dados Aiven
##

db_secret_name = f"{user}-d5hive-aiven-super-1.json"

dbcreds_json = userdata.get(db_secret_name)

if dbcreds_json is None:
    raise ValueError(f"Secret não encontrado no Colab: {db_secret_name}")

dbcreds = json.loads(dbcreds_json)

DB_HOST = dbcreds["dest_host"]
DB_PORT = int(dbcreds["port"])
DB_NAME = dbcreds["database"]
DB_USER = dbcreds["username"]
DB_PASSWORD = dbcreds["password"]

print("Secret da base de dados Aiven carregado com sucesso.")
print("Secret usado:", db_secret_name)
print("DB_HOST:", DB_HOST)
print("DB_PORT:", DB_PORT)
print("DB_NAME:", DB_NAME)
print("DB_USER:", DB_USER)

Secret da base de dados Aiven carregado com sucesso.
Secret usado: acb-d5hive-aiven-super-1.json
DB_HOST: d5hive-super-aiven-1-dfivehive-a1dd.c.aivencloud.com
DB_PORT: 19861
DB_NAME: d5hive
DB_USER: anacarolina


In [4]:
## Ligação à base de dados
##

def ligar_bd():
    try:
        connection = pymysql.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            cursorclass=pymysql.cursors.DictCursor,
            charset="utf8mb4",
            connect_timeout=timeout,
            read_timeout=timeout,
            write_timeout=timeout,
            autocommit=True
        )

        print("Ligação à base de dados com sucesso!")
        clts.elapt["Database connection successful ✅"] = clts.deltat(tstart)
        return connection

    except Exception as e:
        print("Erro na ligação à base de dados:", e)
        clts.elapt[f"Database connection error ❌: {e}"] = clts.deltat(tstart)
        return None

In [5]:
## Mostrar tabelas e validar tabelas escolhidas
##

def listar_tabelas(conn):
    try:
        cursor = conn.cursor()

        query = """
        SELECT table_name AS nome_tabela
        FROM information_schema.tables
        WHERE table_schema = %s
        ORDER BY table_name;
        """

        cursor.execute(query, (DB_NAME,))
        tabelas = cursor.fetchall()

        print("\nTabelas existentes na base de dados:\n")

        if not tabelas:
            print("Não foram encontradas tabelas.")
            return []

        nomes_tabelas = []

        for i, row in enumerate(tabelas, start=1):
            nome_tabela = row["nome_tabela"]
            nomes_tabelas.append(nome_tabela)

            query_colunas = """
            SELECT COUNT(*) AS nr_colunas
            FROM information_schema.columns
            WHERE table_schema = %s
              AND table_name = %s;
            """

            cursor.execute(query_colunas, (DB_NAME, nome_tabela))
            resultado = cursor.fetchone()
            nr_colunas = resultado["nr_colunas"]

            print(f"{i}. {nome_tabela} - {nr_colunas} colunas")

        clts.elapt["Tables listed successfully ✅"] = clts.deltat(tstart)
        return nomes_tabelas

    except Exception as e:
        print("Erro ao listar tabelas:", e)
        clts.elapt[f"List tables error ❌: {e}"] = clts.deltat(tstart)
        return []


def validar_tabelas_escolhidas(conn, tabelas_escolhidas):
    try:
        tabelas_existentes = listar_tabelas(conn)

        tabelas_validas = []
        tabelas_invalidas = []

        for tabela in tabelas_escolhidas:
            if tabela in tabelas_existentes:
                tabelas_validas.append(tabela)
            else:
                tabelas_invalidas.append(tabela)

        print("\nTabelas escolhidas válidas:")
        if tabelas_validas:
            for tabela in tabelas_validas:
                print("-", tabela)
        else:
            print("Nenhuma tabela válida.")

        if tabelas_invalidas:
            print("\nTabelas escolhidas que não existem na BD:")
            for tabela in tabelas_invalidas:
                print("-", tabela)

        return tabelas_validas, tabelas_invalidas

    except Exception as e:
        print("Erro ao validar tabelas escolhidas:", e)
        return [], []

In [6]:
## Funções auxiliares de limpeza e formatação
##

def limpar_nome_ficheiro(nome):
    nome = str(nome).strip().lower()
    nome = re.sub(r"[^a-z0-9_\-\s]", "_", nome)
    nome = re.sub(r"[\s]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)

    if nome == "":
        nome = "dataset"

    return nome


def slugify(texto):
    texto = str(texto).strip().lower()
    texto = re.sub(r"[^a-z0-9_\-\s]", "", texto)
    texto = re.sub(r"[\s_]+", "-", texto)
    texto = re.sub(r"-+", "-", texto)

    if texto == "":
        texto = "dataset"

    return texto


def limpar_valor_json(valor):
    if valor is None:
        return None

    if isinstance(valor, (datetime, date)):
        return valor.isoformat()

    if isinstance(valor, decimal.Decimal):
        return float(valor)

    if isinstance(valor, (dict, list)):
        return valor

    try:
        if pd.isna(valor):
            return None
    except Exception:
        pass

    return valor


def normalizar_dataframe_para_exportacao(df):
    df = df.copy()

    for coluna in df.columns:
        df[coluna] = df[coluna].map(limpar_valor_json)

    return df


def sql_identifier(nome):
    return "`" + str(nome).replace("`", "``") + "`"

In [7]:
## Obter informação das colunas
##

def obter_colunas(conn, nome_tabela):
    try:
        cursor = conn.cursor()

        query = """
        SELECT
            column_name AS nome_coluna,
            data_type AS tipo_sql,
            is_nullable AS permite_nulo,
            column_key AS chave,
            column_comment AS comentario
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name = %s
        ORDER BY ordinal_position;
        """

        cursor.execute(query, (DB_NAME, nome_tabela))
        rows = cursor.fetchall()

        df_colunas = pd.DataFrame(rows)

        return df_colunas

    except Exception as e:
        print(f"Erro ao obter colunas da tabela {nome_tabela}:", e)
        return pd.DataFrame()

In [8]:
## Ler uma tabela da BD para DataFrame
##

def tabela_para_dataframe(conn, nome_tabela, limite=None):
    try:
        cursor = conn.cursor()

        tabela_sql = sql_identifier(nome_tabela)

        if limite is None:
            query = f"SELECT * FROM {tabela_sql};"
        else:
            query = f"SELECT * FROM {tabela_sql} LIMIT {int(limite)};"

        cursor.execute(query)
        rows = cursor.fetchall()

        if rows:
            df = pd.DataFrame(rows)
        else:
            query_colunas = f"SHOW COLUMNS FROM {tabela_sql};"
            cursor.execute(query_colunas)
            colunas_info = cursor.fetchall()
            nomes_colunas = [col["Field"] for col in colunas_info]
            df = pd.DataFrame(columns=nomes_colunas)

        df = normalizar_dataframe_para_exportacao(df)

        return df

    except Exception as e:
        print(f"Erro ao ler a tabela {nome_tabela} para DataFrame:", e)
        return None

In [9]:
## Inferir tipos para HuWise
##

def inferir_tipo_huwise(tipo_sql):
    tipo = str(tipo_sql).lower()

    if tipo in ["tinyint"]:
        return "boolean_or_int"

    if tipo in ["int", "integer", "bigint", "smallint", "mediumint"]:
        return "int"

    if tipo in ["decimal", "numeric", "float", "double", "real"]:
        return "double"

    if tipo in ["date"]:
        return "date"

    if tipo in ["datetime", "timestamp"]:
        return "datetime"

    if tipo in ["time"]:
        return "time"

    if tipo in ["json"]:
        return "json"

    return "text"


def construir_campos_huwise(df_colunas):
    campos = []

    if df_colunas.empty:
        return campos

    for _, row in df_colunas.iterrows():
        campo = {
            "name": row["nome_coluna"],
            "label": str(row["nome_coluna"]).replace("_", " ").title(),
            "source_type": row["tipo_sql"],
            "suggested_huwise_type": inferir_tipo_huwise(row["tipo_sql"]),
            "nullable": row["permite_nulo"] == "YES",
            "primary_key": row["chave"] == "PRI",
            "description": row["comentario"] if row["comentario"] != "" else None
        }

        campos.append(campo)

    return campos

In [10]:
## Overrides opcionais de metadados
##

# Só preencher este dicionário quando for mesmo necessário corrigir algum campo manualmente.
METADADOS_OVERRIDE = {
    # Exemplo opcional:
    # "catalog": {
    #     "title": "Catálogo de datasets municipais",
    #     "publisher": "Município"
    # }
}


def obter_metadados_do_catalog(conn, nome_tabela):
    #A correspondência é feita com base no nome da tabela/dataset. Se não encontrar nada, devolve um dicionário vazio.

    try:
        cursor = conn.cursor()

        query = """
        SELECT *
        FROM catalog
        WHERE alias = %s
           OR id = %s
           OR source = %s
           OR uri = %s
        LIMIT 1;
        """

        cursor.execute(
            query,
            (
                nome_tabela,
                nome_tabela,
                nome_tabela,
                nome_tabela
            )
        )

        row = cursor.fetchone()

        if row is None:
            return {}

        metadata = {
            "dataset_id": row.get("alias") or row.get("id") or slugify(nome_tabela),
            "title": row.get("title"),
            "description": row.get("description"),
            "publisher": row.get("publisher"),
            "contact_point": row.get("contact_point"),
            "theme": row.get("theme"),
            "license": row.get("license"),
            "access_rights": row.get("access_rights"),
            "conforms_to": row.get("conforms_to"),
            "issued": limpar_valor_json(row.get("issued")),
            "modified": limpar_valor_json(row.get("modified")),
            "language": row.get("language") or "pt",
            "dataset_spatial": row.get("dataset_spatial"),
            "temporal_start": limpar_valor_json(row.get("temporal_start")),
            "temporal_end": limpar_valor_json(row.get("temporal_end")),
            "update_frequency": row.get("update_frequency"),
            "version": row.get("version"),
            "version_notes": row.get("version_notes"),
            "provenance": row.get("provenance"),
            "source": row.get("source"),
            "relation": row.get("relation"),
            "applicable_legislation": row.get("applicable_legislation"),
            "hvd_category": row.get("hvd_category"),
            "is_referenced_by": row.get("is_referenced_by")
        }

        metadata = {
            chave: valor
            for chave, valor in metadata.items()
            if valor not in [None, "", []]
        }

        return metadata

    except Exception as e:
        print(f"Não foi possível obter metadados automáticos para {nome_tabela}: {e}")
        return {}

In [11]:
## Construir metadados para HuWise
##

def construir_metadados_huwise(conn, nome_tabela, df, df_colunas):

    metadados_catalog = obter_metadados_do_catalog(conn, nome_tabela)
    override = METADADOS_OVERRIDE.get(nome_tabela, {})

    dataset_id = (
        override.get("dataset_id")
        or metadados_catalog.get("dataset_id")
        or slugify(nome_tabela)
    )

    metadata = {
        "dataset_id": dataset_id,
        "source_table": nome_tabela,

        "title": (
            override.get("title")
            or metadados_catalog.get("title")
            or nome_tabela.replace("_", " ").title()
        ),

        "description": (
            override.get("description")
            or metadados_catalog.get("description")
            or f"Dataset gerado automaticamente a partir da tabela {nome_tabela}."
        ),

        "theme": override.get("theme") or metadados_catalog.get("theme"),
        "publisher": override.get("publisher") or metadados_catalog.get("publisher"),
        "license": override.get("license") or metadados_catalog.get("license"),
        "language": override.get("language") or metadados_catalog.get("language") or "pt",

        "keywords": override.get("keywords") or metadados_catalog.get("keywords", []),

        "contact_point": override.get("contact_point") or metadados_catalog.get("contact_point"),
        "access_rights": override.get("access_rights") or metadados_catalog.get("access_rights"),
        "conforms_to": override.get("conforms_to") or metadados_catalog.get("conforms_to"),
        "issued": override.get("issued") or metadados_catalog.get("issued"),
        "modified": override.get("modified") or metadados_catalog.get("modified"),
        "dataset_spatial": override.get("dataset_spatial") or metadados_catalog.get("dataset_spatial"),
        "temporal_start": override.get("temporal_start") or metadados_catalog.get("temporal_start"),
        "temporal_end": override.get("temporal_end") or metadados_catalog.get("temporal_end"),
        "update_frequency": override.get("update_frequency") or metadados_catalog.get("update_frequency"),
        "version": override.get("version") or metadados_catalog.get("version"),
        "version_notes": override.get("version_notes") or metadados_catalog.get("version_notes"),
        "provenance": override.get("provenance") or metadados_catalog.get("provenance"),
        "source": override.get("source") or metadados_catalog.get("source"),
        "relation": override.get("relation") or metadados_catalog.get("relation"),
        "applicable_legislation": (
            override.get("applicable_legislation")
            or metadados_catalog.get("applicable_legislation")
        ),
        "hvd_category": override.get("hvd_category") or metadados_catalog.get("hvd_category"),
        "is_referenced_by": override.get("is_referenced_by") or metadados_catalog.get("is_referenced_by"),

        "row_count_exported": int(len(df)),
        "column_count": int(len(df.columns)),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "fields": construir_campos_huwise(df_colunas)
    }

    metadata = {
        chave: valor
        for chave, valor in metadata.items()
        if valor not in [None, "", []]
    }

    return metadata

In [12]:
## Validação dos dados e metadados
##

CAMPOS_METADATA_RECOMENDADOS = [
    "title",
    "description",
    "theme",
    "publisher",
    "license",
    "language"
]


def validar_pacote_huwise(metadata, df):
    erros = []
    avisos = []

    for campo in CAMPOS_METADATA_RECOMENDADOS:
        valor = metadata.get(campo)

        if valor is None or valor == "" or valor == []:
            avisos.append(f"Metadado por preencher/rever: {campo}")

    if df is None:
        erros.append("DataFrame não foi criado.")
        return {
            "status": "erro",
            "errors": erros,
            "warnings": avisos
        }

    if df.empty:
        avisos.append("A tabela não tem linhas.")

    if len(df.columns) == 0:
        erros.append("A tabela não tem colunas.")

    nomes_colunas = list(df.columns)

    if len(nomes_colunas) != len(set(nomes_colunas)):
        erros.append("Existem nomes de colunas repetidos.")

    status = "erro" if erros else "ok"

    return {
        "status": status,
        "errors": erros,
        "warnings": avisos
    }

In [13]:
## Preparar uma tabela para HuWise sem enviar
##

def preparar_tabela_para_huwise(conn, nome_tabela, export_dir=EXPORT_DIR, limite=None):
    print(f"\nA preparar tabela para HuWise: {nome_tabela}")

    try:
        df = tabela_para_dataframe(conn, nome_tabela, limite=limite)

        if df is None:
            raise Exception("Falha na leitura da tabela")

        df_colunas = obter_colunas(conn, nome_tabela)

        if verbose:
            print(df.head())
            print("shape:", df.shape)

        metadata = construir_metadados_huwise(conn, nome_tabela, df, df_colunas)
        validacao = validar_pacote_huwise(metadata, df)

        dataset_id = metadata["dataset_id"]
        nome_base = limpar_nome_ficheiro(dataset_id)

        csv_path = export_dir / f"{nome_base}.csv"
        metadata_path = export_dir / f"{nome_base}_metadata.json"
        package_path = export_dir / f"{nome_base}_package.json"

        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        package = {
            "dry_run": True,
            "action": "prepare_for_huwise",
            "dataset_id": dataset_id,
            "source_table": nome_tabela,
            "files": {
                "data_csv": str(csv_path),
                "metadata_json": str(metadata_path)
            },
            "metadata": metadata,
            "validation": validacao,
            "note": "Este package ainda não foi enviado para a HuWise. Serve apenas para revisão e futura integração."
        }

        with open(package_path, "w", encoding="utf-8") as f:
            json.dump(package, f, ensure_ascii=False, indent=2)

        print(f"Tabela preparada com sucesso: {nome_tabela}")
        print("CSV:", csv_path)
        print("Metadata JSON:", metadata_path)
        print("Package JSON:", package_path)
        print("Estado:", validacao["status"])

        if validacao["warnings"]:
            print("Avisos:")
            for aviso in validacao["warnings"]:
                print("-", aviso)

        if validacao["errors"]:
            print("Erros:")
            for erro in validacao["errors"]:
                print("-", erro)

        return {
            "tabela": nome_tabela,
            "dataset_id": dataset_id,
            "linhas": len(df),
            "colunas": len(df.columns),
            "estado": validacao["status"],
            "erros": len(validacao["errors"]),
            "avisos": len(validacao["warnings"]),
            "csv_path": str(csv_path),
            "metadata_path": str(metadata_path),
            "package_path": str(package_path)
        }

    except Exception as e:
        print(f"Erro ao preparar tabela {nome_tabela} para HuWise:", e)

        return {
            "tabela": nome_tabela,
            "dataset_id": None,
            "linhas": None,
            "colunas": None,
            "estado": f"erro: {e}",
            "erros": 1,
            "avisos": None,
            "csv_path": None,
            "metadata_path": None,
            "package_path": None
        }

In [14]:
def preparar_bd_para_huwise(conn, tabelas_escolhidas):
    try:
        tabelas_validas, tabelas_invalidas = validar_tabelas_escolhidas(
            conn,
            tabelas_escolhidas
        )

        if not tabelas_validas:
            print("Não existem tabelas válidas para preparar.")
            return None

        resumo = []

        for nome_tabela in tabelas_validas:
            resultado = preparar_tabela_para_huwise(
                conn,
                nome_tabela,
                EXPORT_DIR,
                max_rows_per_table
            )

            resumo.append(resultado)

        df_resumo = pd.DataFrame(resumo)

        resumo_path = EXPORT_DIR / "resumo_exportacao.csv"
        df_resumo.to_csv(resumo_path, index=False, encoding="utf-8-sig")

        manifest = {
            "dry_run": True,
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "database": DB_NAME,
            "number_of_tables": len(resumo),
            "tables_successfully_prepared": int((df_resumo["estado"] == "ok").sum()),
            "tables": resumo,
            "note": "Manifesto de preparação local. Nada foi enviado para a HuWise."
        }

        manifest_path = EXPORT_DIR / "manifest.json"

        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)

        print("\nResumo da preparação para HuWise:")
        display(df_resumo)

        print("Manifest criado:", manifest_path)
        print("Resumo CSV criado:", resumo_path)

        return df_resumo

    except Exception as e:
        print("Erro ao preparar base de dados para HuWise:", e)
        return None

In [15]:
## Execução
##

conn = None

try:
    conn = ligar_bd()

    if conn is not None:
        df_resumo = preparar_bd_para_huwise(
            conn,
            TABELAS_ESCOLHIDAS
        )

finally:
    if conn is not None:
        conn.close()
        print("Ligação à base de dados fechada.")

Ligação à base de dados com sucesso!

Tabelas existentes na base de dados:

1. api_log - 8 colunas
2. catalog - 28 colunas
3. catalog_distribution - 18 colunas
4. catalog_internal - 10 colunas
5. catalog_internal_v2 - 10 colunas
6. catalog_keyword - 3 colunas
7. catalog_v2 - 28 colunas
8. gjson - 3 colunas
9. icao_obs - 17 colunas
10. veolia_consumos - 10 colunas

Tabelas escolhidas válidas:
- catalog
- catalog_v2
- catalog_distribution
- catalog_internal
- catalog_internal_v2
- catalog_keyword

A preparar tabela para HuWise: catalog
   id       uri alias                                              title  \
0   1   meteo01  None  Estação meteorológica da Cobertura verde do Fo...   
1   2  qualar01  None         Estação de qualidade do ar (FORUM DA MAIA)   
2   3  qualar02  None     Estação de qualidade do ar (Rua do Património)   
3   4  qualar03  None  Estação de qualidade do ar (R. Eng. Duarte Pac...   
4   5   ruido01  None                         Medidor de ruído (Local 1)   

   

,tabela,dataset_id,linhas,colunas,estado,erros,avisos,csv_path,metadata_path,package_path
0,catalog,catalog,28,28,ok,0,3,huwise_export/catalog.csv,huwise_export/catalog_metadata.json,huwise_export/catalog_package.json
1,catalog_v2,catalog-v2,28,28,ok,0,3,huwise_export/catalog-v2.csv,huwise_export/catalog-v2_metadata.json,huwise_export/catalog-v2_package.json
2,catalog_distribution,catalog-distribution,2,18,ok,0,3,huwise_export/catalog-distribution.csv,huwise_export/catalog-distribution_metadata.json,huwise_export/catalog-distribution_package.json
3,catalog_internal,catalog-internal,24,10,ok,0,3,huwise_export/catalog-internal.csv,huwise_export/catalog-internal_metadata.json,huwise_export/catalog-internal_package.json
4,catalog_internal_v2,catalog-internal-v2,24,10,ok,0,3,huwise_export/catalog-internal-v2.csv,huwise_export/catalog-internal-v2_metadata.json,huwise_export/catalog-internal-v2_package.json
5,catalog_keyword,catalog-keyword,9,3,ok,0,3,huwise_export/catalog-keyword.csv,huwise_export/catalog-keyword_metadata.json,huwise_export/catalog-keyword_package.json


Manifest criado: huwise_export/manifest.json
Resumo CSV criado: huwise_export/resumo_exportacao.csv
Ligação à base de dados fechada.


In [16]:
## Criar ZIP para download
##

try:
    zip_path = shutil.make_archive(export_folder, "zip", EXPORT_DIR)

    print("ZIP criado com sucesso:", zip_path)

    files.download(zip_path) # Descarreg automaticamente o zip

except Exception as e:
    print("Erro ao criar ZIP:", e)

ZIP criado com sucesso: /content/huwise_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>